In [99]:
import torch
import torch.nn.functional as f
import matplotlib.pyplot as plt
%matplotlib inline

In [100]:
words = open('names.txt','r').read().splitlines()
words[:4]

['emma', 'olivia', 'ava', 'isabella']

In [101]:
chars = sorted(list(set("".join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.']=0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
vocab_size

27

In [102]:
block_size = 3
def dataset(words):

    X,y = [],[]
    for w in words:
        context = [0]*3
        ch = w+'.'
        for c in ch:
            ix = stoi[c]
            X.append(context)
            y.append(ix)
            context=context[1:]+[ix]
            if ix==0:
                break
    
    X = torch.tensor(X).to('mps')
    y = torch.tensor(y).to('mps')
    print(X.shape,y.shape)
    return X,y

import random
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))
Xtr, ytr = dataset(words[:n1])
Xdev,ydev = dataset(words[n1:n2])
Xte,yte = dataset(words[n2:])

torch.Size([182523, 3]) torch.Size([182523])
torch.Size([22628, 3]) torch.Size([22628])
torch.Size([22995, 3]) torch.Size([22995])


In [109]:
n_emb = 10
n_hidden = 200

g = torch.Generator().manual_seed(3)
C = torch.randn((vocab_size,n_emb),generator=g).to('mps') 
W1 = torch.randn((n_emb*block_size, n_hidden), generator=g).to('mps') * (5/3)/((n_emb*block_size)**0.5)
b1 = torch.randn((n_hidden,), generator=g).to('mps') *0.01
W2 = torch.randn((n_hidden, vocab_size),generator=g).to('mps') *0.01
b2 = torch.randn((vocab_size,),generator=g).to('mps') *0.1
bngain= torch.ones((1,n_hidden)).to('mps')
bnbias = torch.zeros((1,n_hidden)).to('mps')
bnmean_running = torch.zeros((1,n_hidden)).to('mps')
bnstd_running = bnbias = torch.ones((1,n_hidden)).to('mps')

parameters = [C,W1,b1,W2,b2, bnbias, bngain]
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad=True
    

12297


In [110]:
max_steps = 200_000
batch_size = 32
lossi=[]

for i in range(max_steps):
    ix = torch.randint(0,Xtr.shape[0], (batch_size,), generator=g).to('mps')
    Xb,yb = Xtr[ix], ytr[ix]
    
    emb = C[Xb]
    embcat = emb.view(emb.shape[0],-1)
    h_ = embcat@W1 
    bnmeani = h_.mean(0,keepdim=True)
    bnstdi = h_.std(0,keepdim=True)
    h_ = (bngain*(h_-bnmeani)/bnstdi)+ bnbias

    with torch.no_grad():
        bnmean_running = 0.999*bnmean_running+0.001*bnmeani
        bnstd_running = 0.999*bnstd_running+0.001*bnstdi

    h = torch.tanh(h_)
    logits = h@W2+b2
    loss = f.cross_entropy(logits,yb)

    for p in parameters:
        p.grad=None
    loss.backward()

    lr = 0.1 if i<100_000 else 0.01
    for p in parameters:
        p.data +=-lr*p.grad

    if i%10000==0:
        print(f'{i}/{max_steps}: {loss.item():.4f}')
    lossi.append(loss.log10().item())


0/200000: 3.3361
10000/200000: 2.6432
20000/200000: 2.3202
30000/200000: 1.9269
40000/200000: 2.3298
50000/200000: 1.7317
60000/200000: 2.0334
70000/200000: 2.4120
80000/200000: 1.9108
90000/200000: 2.2893
100000/200000: 2.2114
110000/200000: 2.0523
120000/200000: 1.8689
130000/200000: 1.8345
140000/200000: 1.7134
150000/200000: 2.2944
160000/200000: 2.0318
170000/200000: 2.3746
180000/200000: 2.4599
190000/200000: 2.1007


In [112]:
bngain, bnmean_running

(tensor([[1.3571, 1.1067, 1.1435, 1.2560, 1.3630, 1.0997, 1.5084, 1.0735, 1.4182,
          1.1557, 1.7511, 1.2762, 0.9946, 1.4351, 1.4835, 1.2368, 1.3611, 1.4937,
          1.1509, 1.1136, 1.2319, 1.4398, 1.3433, 1.1739, 1.3709, 1.7014, 0.7652,
          1.5530, 1.5439, 1.4406, 1.3782, 1.2311, 1.1255, 1.2266, 1.2522, 1.4509,
          1.0000, 1.2098, 1.1801, 1.2331, 1.2793, 1.3325, 0.9376, 1.6360, 1.2752,
          0.8851, 1.5489, 1.3955, 1.2240, 1.4812, 1.3603, 1.4283, 1.3788, 1.1653,
          1.3359, 1.0195, 1.1340, 1.2162, 0.7845, 1.9826, 1.4374, 1.4186, 1.1705,
          1.2398, 1.1933, 1.3246, 1.0210, 1.4990, 1.7345, 1.0799, 1.6618, 0.9313,
          1.1211, 1.1719, 0.9711, 0.9984, 1.5905, 1.3364, 1.5357, 1.0298, 1.1040,
          0.8774, 1.5102, 1.2936, 1.5392, 1.3749, 1.1547, 1.5513, 1.3182, 1.1107,
          1.3787, 0.6024, 1.7545, 1.3219, 1.4180, 0.9108, 1.2840, 1.2550, 0.9853,
          1.3459, 1.2882, 0.9011, 1.3187, 1.5908, 1.0618, 1.6154, 0.9882, 1.2577,
          1.3195

In [106]:
@torch.no_grad()
def split_loss(split):
    x,y = {
        'train':[Xtr,ytr],
        "val":[Xdev,ydev],
        'test':[Xte,yte]
    }[split]
    emb = C[x]
    embcat = emb.view(emb.shape[0],-1)
    h_ = embcat@W1+b1
    h_ = (bngain*(h_-h_.mean(0,keepdim=True))/h_.std(0,keepdim=True))+ bnbias
    h = torch.tanh(h_)
    logits = h@W2+b2
    loss = f.cross_entropy(logits,y)
    print(split, loss.item())

split_loss('train')
split_loss('test')



train 2.070422887802124
test 2.104860544204712


In [107]:
g = torch.Generator(device='mps').manual_seed(3)
for _ in range(20):
    out = []
    context = [0]*block_size
    while True:
        emb = C[torch.tensor([context]).to('mps')]
        h = torch.tanh(emb.view(1,-1)@W1+b1).to('mps')
        logits = h@W2+b2
        probs = f.softmax(logits, dim=1)
        ix = torch.multinomial(probs,num_samples=1,generator=g).to('mps').item()
        context = context[1:]+[ix]
        out.append(ix)
        if ix==0:
            break

    print("".join(itos[i] for i in out))

rhygalington.
xvythl.
juvctiston.
shathlarahabglea.
ksof.
xnylannesilijfrixllx.
xzhvik.
zyx.
sylegkorzkeldy.
xington.
jamichrithrgatsenxd.
scodespouxlyxallel.
clekghomphr.
xzhi.
rendzishazger.
xzayvon.
kryn.
zahmbertlmeysaibraxleezhamzulca.
ncyumillley.
nissyfedmahmikkassaxxsy.
